In [ ]:
# !pip install "zenml[server]==0.94.2"
# !pip install pyngrok

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
NGROK_TOKEN = os.getenv("NGROK_TOKEN")
!ngrok authtoken {NGROK_TOKEN}

Authtoken saved to configuration file: C:\Users\YOUSSEF\AppData\Local\ngrok\ngrok.yml


In [3]:
import os, sys, shutil
from pathlib import Path

# full ZenML reset (local + global)
paths_to_remove = [
    Path(".zen"),  # project-local repo
    Path.home() / ".config" / "zenml",  # common global path
    Path(os.path.expandvars(r"%USERPROFILE%\.config\zenml")),
    Path(os.path.expandvars(r"%APPDATA%\zenml")),
]

for p in paths_to_remove:
    if p.exists():
        print("Removing:", p)
        shutil.rmtree(p, ignore_errors=True)

print("Kernel Python:", sys.executable)

# init from terminal to avoid errors

Removing: .zen
Removing: C:\Users\YOUSSEF\AppData\Roaming\zenml
Kernel Python: c:\Users\YOUSSEF\Desktop\Machine Learning\.venv\Scripts\python.exe


In [4]:
import numpy as np
import pandas as pd
from sklearn.base import ClassifierMixin
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

In [5]:
# using zenml steps to define our pipeline blocks
from zenml import step
from typing_extensions import Annotated
from typing import Tuple


# Step 1: load the dataset and split it
@step
def importer() -> Tuple[
    Annotated[np.ndarray, "X_train"],
    Annotated[np.ndarray, "X_test"],
    Annotated[np.ndarray, "y_train"],
    Annotated[np.ndarray, "y_test"],
]:
    digits = load_digits()
    data = digits.images.reshape((len(digits.images), -1))
    
    X_train, X_test, y_train, y_test = train_test_split(
        data, digits.target, test_size=0.2, shuffle=False
    )
    
    return X_train, X_test, y_train, y_test

# Step 2: define the model and train it
@step
def svc_trainer(
    X_train: np.ndarray,
    y_train: np.ndarray
):
    model = SVC(gamma=0.001)
    model.fit(X_train, y_train)
    
    return model


# Step 3: eval model
@step
def eval(
    X_test: np.ndarray,
    y_test: np.ndarray,
    model: ClassifierMixin
):
    test_acc = model.score(X_test, y_test)
    print(f"Test accuracy = {test_acc}")
    
    return test_acc

In [6]:
# Create pipline from steps
from zenml import pipeline

@pipeline
def digits_pipeline():
    # step 1
    X_train, X_test, y_train, y_test = importer() # -> use steps
    
    # step 2
    model = svc_trainer(X_train, y_train)
    
    # step 3
    eval(X_test, y_test, model)

# in terminal
1. zenml init
2. zenml login --blocking

In [7]:
digits_svc_pipeline = digits_pipeline()
# digits_svc_pipeline.run(unlisted = True)

Daemon functionality is currently not supported on Windows.
Initiating a new run for the pipeline: digits_pipeline.
Registered new pipeline: digits_pipeline.
Using user: default
Using stack: default
  deployer: default
  orchestrator: default
  artifact_store: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step importer has started.
Step importer has finished in 2.410s.
Step svc_trainer has started.
Step svc_trainer has finished in 0.211s.
Step eval has started.
Test accuracy = 0.9583333333333334
Step eval has finished in 0.466s.
Pipeline run has finished in 5.117s.
